In [1]:
%load_ext autoreload

%autoreload 2

In [2]:
import pickle
import numpy as np
from typing import Dict, List, Optional, Tuple, Union, Any, Callable, Mapping
import pandas as pd
from rdkit.Chem import rdFingerprintGenerator
from gensim.models.doc2vec import Doc2Vec, TaggedDocument
import lightgbm as lgb
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score
from rdkit.Chem import MACCSkeys
from sklearn.metrics import (f1_score, matthews_corrcoef, balanced_accuracy_score, roc_auc_score, cohen_kappa_score, auc, precision_recall_curve)

In [3]:
def fin(df, radius, fpSize):
    fingerprints = []
    onbits_list = []
    fp_generator = rdFingerprintGenerator.GetMorganGenerator(radius=radius, fpSize=fpSize)
    for i, mol in enumerate(df["ROMol"]):
        try:
            fp = fp_generator.GetFingerprint(mol)
            # 1になっているビットの位置を取得
            onbits = list(fp.GetOnBits())
            onbits_list.append(onbits)
            
            # NumPy配列も必要なら
            fp_np = fp_generator.GetFingerprintAsNumPy(mol)
            fingerprints.append(fp_np)

        except Exception as e:
            print(f"Error processing molecule {i}: {e}")
            continue
    return np.array(fingerprints), onbits_list

def add_vectors(fp_list: List[List[int]], model: Doc2Vec) -> List[np.ndarray]:
    """Combine document vectors based on fingerprints
    
    Args:
        fp_list: List of fingerprint lists, where each fingerprint is represented as a list of indices
        model: Trained Doc2Vec model containing document vectors
        
    Returns:
        List of compound vectors as numpy arrays
    """
    compound_vec = []
    for i in fp_list:
        fingerprint_vec = 0
        for j in i:
            fingerprint_vec += model.dv.vectors[j]
        compound_vec.append(fingerprint_vec)
    return compound_vec

def calculate_metrics(y_true, y_pred, y_proba):

    metrics = {}
    metrics['f1'] = f1_score(y_true, y_pred)
    metrics['mcc'] = matthews_corrcoef(y_true, y_pred)
    metrics['balanced_accuracy'] = balanced_accuracy_score(y_true, y_pred)
    metrics['roc_auc'] = roc_auc_score(y_true, y_proba)
    metrics['kappa'] = cohen_kappa_score(y_true, y_pred)
    precision, recall, _ = precision_recall_curve(y_true, y_proba)
    metrics['pr_auc'] = auc(recall, precision)
    return metrics

In [4]:
def evaluate_category(X_vec: np.ndarray, 
                      y: np.ndarray, 
                      lightgbm_model: lgb.LGBMClassifier
                      ) -> Dict[str, Union[List[float], float]]:
    
    # 全ての評価指標のスコアを格納する辞書
    all_train_scores = {'f1': [], 'mcc': [], 'balanced_accuracy': [], 'roc_auc': [], 'kappa': [], 'pr_auc': []}
    all_test_scores = {'f1': [], 'mcc': [], 'balanced_accuracy': [], 'roc_auc': [], 'kappa': [], 'pr_auc': []}


    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=0)
    for train_idx, test_idx in skf.split(range(len(y)), y):
        X_train_vec, X_test_vec = X_vec[train_idx], X_vec[test_idx]
        y_train, y_test = y[train_idx], y[test_idx]
        
        lightgbm_model.fit(X_train_vec, y_train)
        y_train_pred = lightgbm_model.predict(X_train_vec)
        y_test_pred = lightgbm_model.predict(X_test_vec)
        y_train_proba = lightgbm_model.predict_proba(X_train_vec)[:, 1]
        y_test_proba = lightgbm_model.predict_proba(X_test_vec)[:, 1]

        train_metrics = calculate_metrics(y_train, y_train_pred, y_train_proba)
        test_metrics = calculate_metrics(y_test, y_test_pred, y_test_proba)
        for metric_name in all_train_scores.keys():
            all_train_scores[metric_name].append(train_metrics[metric_name])
            all_test_scores[metric_name].append(test_metrics[metric_name])
    
    # 結果を整理
    results = {}
    for metric_name in all_train_scores.keys():
        results[metric_name] = {
            'train_scores': all_train_scores[metric_name],
            'test_scores': all_test_scores[metric_name],
            'mean_train': np.mean(all_train_scores[metric_name]),
            'mean_test': np.mean(all_test_scores[metric_name])
        }
    
    return results

def build_doc2vec_model(corpus: List[List[str]], 
                        tag_list: List[List[int]], #変更
                        doc2vec_param: Dict[str, Any]) -> Doc2Vec:
    """
    Build and train a Doc2Vec model from corpus and structure information
    
    Args:
        corpus: List of lists containing tokenized text for each document
        list: List of lists containing tags for each document
        doc2vec_param: Dictionary of parameters for the Doc2Vec model
        
    Returns:
        Trained Doc2Vec model
    """
    tagged_documents = [
        TaggedDocument(words=corpus, tags=tag_list[i]) #変更
        for i, corpus in enumerate(corpus)
    ]
    
    model = Doc2Vec(tagged_documents, **doc2vec_param)
    
    return model

def main(df: pd.DataFrame, tag_list: List[Any], #変更
         doc2vec_param: Dict[str, Any], 
         lightgbm_model: lgb.LGBMClassifier,
         purpose_description: str,) -> Dict[str, Dict[str, float]]:
    """
    Main function to train and evaluate compound classification models using provided features and Doc2Vec.
    
    Args:
        input_path: Path to the pickle file containing compound data
        feature_list: List of molecular features (like fingerprints) to use in the model
        doc2vec_param: Parameters for the Doc2Vec model
        lightgbm_model: Pre-configured LightGBM classifier
        purpose_description: Column name in the DataFrame containing text descriptions
        
    Returns:
        Dictionary mapping category names to evaluation results
    """
        
    # Define categories to evaluate
    categories = [
        'antioxidant', 'anti_inflammatory_agent', 'allergen', 'dye', 'toxin', 
        'flavouring_agent', 'agrochemical', 'volatile_oil', 'antibacterial_agent', 'insecticide'
    ]
    
    # Prepare corpus for Doc2Vec
    corpus = df[purpose_description].tolist()#変更
    
    # Build Doc2Vec model
    model = build_doc2vec_model(corpus, tag_list, doc2vec_param)
    
    # Generate compound vectors
    compound_vec = add_vectors(tag_list, model)
    X_vec = np.array([compound_vec[i] for i in range(len(df))])
    
    
    # Evaluate each category
    results = {}
    for category in categories:
        y = np.array([1 if i == category else 0 for i in df[category]])
        results[category] = evaluate_category(category, X_vec, y, lightgbm_model)

    return results

In [103]:
# Example usage - replace with your actual params
import warnings
warnings.filterwarnings('ignore', message='X does not have valid feature names')
doc2vec_param: Dict[str, Any] = {
    'vector_size': 150,
    'dm': 1,
    'window': 9,
    'min_count': 0,
    'alpha': 0.014996471116783728,
    'sample': 4.4713728630733355e-05,
    'epochs': 840,
    'negative': 12,
    'workers': 1,
    'seed': 0
}

gbm_params: Dict[str, Any] = {
    "boosting_type": "dart",
    "num_leaves": 48,
    "max_depth": 5,
    "learning_rate": 0.04166324251391809,
    "n_estimators": 736,
    "class_weight": "balanced",
    "min_split_gain": 0.009346925180781129,
    "min_child_weight": 0.0007929549087822909,
    "min_child_samples": 37,
    "reg_alpha": 1.757104268180148,
    "reg_lambda": 1.463369722508726,
    "feature_fraction": 0.50163362868711,
    "feature_fraction_bynode": 0.8321043377994284,
    "subsample": 0.6974385909748512,
    "colsample_bytree": 0.6568268046410831,
    "subsample_freq": 5,
    "drop_rate": 0.24668126335938073,
    "max_drop": 28,
    "skip_drop": 0.5591506516119614,
    "uniform_drop": True,
    "xgboost_dart_mode": True,
    "objective": "binary",
    "random_state": 0,
    "verbose": -1,
    "force_col_wise": True
}

# Create classifier
lightgbm_model = lgb.LGBMClassifier(**gbm_params)

# Load dataset for later use
# Example usage - replace with your actual file paths
input_path = "data/train_df2.pkl"
with open(input_path, "rb") as f:
    df = pickle.load(f)

# Tag 2048ECFP
fp2048_list, bit_list = fin(df, 2, 2048)
results = main(df, bit_list, doc2vec_param, lightgbm_model, "description_gensim")

In [140]:
with open("result_tagchange/ECFP2048bit.pkl", "wb") as f:
    pickle.dump(results, f)

In [139]:
categories = [
        'antioxidant', 'anti_inflammatory_agent', 'allergen', 'dye', 'toxin', 
        'flavouring_agent', 'agrochemical', 'volatile_oil', 'antibacterial_agent', 'insecticide'
    ]
li = []
for category, result in results.items():
    print(f"## {category} ##")
    print(results[category]['mcc']["mean_test"])
    li.append(results[category]['mcc']["mean_test"])
print("")
print(np.mean(li))

## antioxidant ##
0.6695757205435798
## anti_inflammatory_agent ##
0.6618773566035163
## allergen ##
0.625385987037887
## dye ##
0.9094083785053367
## toxin ##
0.5804947009552295
## flavouring_agent ##
0.674153228416998
## agrochemical ##
0.7664249219934052
## volatile_oil ##
0.7838213586656304
## antibacterial_agent ##
0.623354476212082
## insecticide ##
0.7253445358697646

0.7019840664803428


!!! ECFP 4096 !!!

In [ ]:
# Tag 4096ECFP
fp4096_list, bit_list = fin(df, 3, 4096)
results4096 = main(df, bit_list, doc2vec_param, lightgbm_model, "description_gensim")

In [138]:
with open("result_tagchange/ECFP4096bit.pkl", "wb") as f:
    pickle.dump(results4096, f)

In [117]:
categories = [
        'antioxidant', 'anti_inflammatory_agent', 'allergen', 'dye', 'toxin', 
        'flavouring_agent', 'agrochemical', 'volatile_oil', 'antibacterial_agent', 'insecticide'
    ]
li = []
for category, result in results4096.items():
    print(f"## {category} ##")
    print(results4096[category]['mcc']["mean_test"])
    li.append(results4096[category]['mcc']["mean_test"])
print("")
print(np.mean(li))

## antioxidant ##
0.6825388153946739
## anti_inflammatory_agent ##
0.6956165157526341
## allergen ##
0.6619015583523324
## dye ##
0.9267184241965426
## toxin ##
0.6174931945910888
## flavouring_agent ##
0.7180149889064215
## agrochemical ##
0.7893064227493397
## volatile_oil ##
0.7944544906364606
## antibacterial_agent ##
0.6347927623112386
## insecticide ##
0.7548672243802339

0.7275704397270966


!!!! MACCS keys !!!

In [129]:
def generate_maccs_fingerprints(df: pd.DataFrame) -> Tuple[List[Optional[List[int]]], List[int]]:
    """
    Generate MACCS fingerprints for molecules in the dataframe
    
    Args:
        df: DataFrame containing RDKit molecule objects in the 'ROMol' column
        
    Returns:
        Tuple containing:
        - List of fingerprints (lists of on-bit indices) or None for invalid molecules
        - List of indices of invalid molecules in the original dataframe
    """
    maccs_features = []
    invalid_indices = []
    
    for idx, mol in enumerate(df["ROMol"]):
        fps = MACCSkeys.GenMACCSKeys(mol)
        fp_bits = list(fps.GetOnBits())
        
        if len(fp_bits) == 0:
            print(f"Invalid MACCS for index {idx}")
            maccs_features.append(None)
            invalid_indices.append(idx)
        else:
            maccs_features.append(fp_bits)
            
    return maccs_features, invalid_indices

def create_index_mapping(df_length: int, invalid_indices: List[int]) -> Dict[int, int]:
    """
    Create a mapping from original dataframe indices to filtered dataframe indices
    
    Args:
        df_length: Length of the original dataframe
        invalid_indices: List of indices to exclude from the mapping
        
    Returns:
        Dictionary mapping original indices to new filtered indices
    """
    original_to_filtered = {}
    filtered_idx = 0
    
    for orig_idx in range(df_length):
        if orig_idx not in invalid_indices:
            original_to_filtered[orig_idx] = filtered_idx
            filtered_idx += 1
            
    return original_to_filtered

def evaluate_with_keys(category: str,
                       X_vec: np.ndarray, 
                       y: np.ndarray, 
                       y_keys: np.ndarray, 
                       lightgbm_model: lgb.LGBMClassifier,
                       index_mapping: Dict[int, int]) -> Dict[str, Union[List[float], float]]:
    #変更全部

    all_train_scores = {'f1': [], 'mcc': [], 'balanced_accuracy': [], 'roc_auc': [], 'kappa': [], 'pr_auc': []}
    all_test_scores = {'f1': [], 'mcc': [], 'balanced_accuracy': [], 'roc_auc': [], 'kappa': [], 'pr_auc': []}


    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=0)
    for train_idx, test_idx in skf.split(range(len(df)), y):
        # Convert indices to MACCS-compatible indices
        train_idx_keys = [index_mapping[idx] for idx in train_idx if idx in index_mapping]
        test_idx_keys = [index_mapping[idx] for idx in test_idx if idx in index_mapping]
         
        # Extract training and testing data
        X_train_vec, X_test_vec = X_vec[train_idx_keys], X_vec[test_idx_keys]
        y_train, y_test = y_keys[train_idx_keys], y_keys[test_idx_keys]
        # Train model and make predictions
        lightgbm_model.fit(X_train_vec, y_train)
        y_train_pred = lightgbm_model.predict(X_train_vec)
        y_test_pred = lightgbm_model.predict(X_test_vec)
        y_train_proba = lightgbm_model.predict_proba(X_train_vec)[:, 1]
        y_test_proba = lightgbm_model.predict_proba(X_test_vec)[:, 1]

        train_metrics = calculate_metrics(y_train, y_train_pred, y_train_proba)
        test_metrics = calculate_metrics(y_test, y_test_pred, y_test_proba)
        for metric_name in all_train_scores.keys():
            all_train_scores[metric_name].append(train_metrics[metric_name])
            all_test_scores[metric_name].append(test_metrics[metric_name])

    results = {}
    for metric_name in all_train_scores.keys():
        results[metric_name] = {
            'train_scores': all_train_scores[metric_name],
            'test_scores': all_test_scores[metric_name],
            'mean_train': np.mean(all_train_scores[metric_name]),
            'mean_test': np.mean(all_test_scores[metric_name])
        }

    return results

def main_keys(input_file: str, maccs_features, invalid_indices, params: Dict[str, Any], lightgbm_model: lgb.LGBMClassifier, 
               purpose_description: str) -> Dict[str, Dict[str, float]]:
    """
    Train and evaluate models using MACCS key fingerprints.
    
    Args:
        input_file: Path to the pickle file containing compound data
        params: Parameters for the Doc2Vec model
        lightgbm_model: Pre-configured LightGBM classifier
        purpose_description: Column name in the DataFrame containing text descriptions
        
    Returns:
        Dictionary mapping category names to evaluation results
    """
    # Load dataset
    with open(input_file, "rb") as f:
        df = pickle.load(f)

    # Define categories
    categories = [
        'antioxidant', 'anti_inflammatory_agent', 'allergen', 'dye', 'toxin',
        'flavouring_agent', 'agrochemical', 'volatile_oil', 'antibacterial_agent', 'insecticide'
    ]

    # Create a filtered dataframe with valid MACCS fingerprints
    df_keys = df.copy()
    df_keys["keys"] = maccs_features
    df_keys = df_keys.dropna(subset=['keys'])
    print(f"Number of compounds with valid keys: {len(df_keys)}")
    
    # Prepare data for Doc2Vec
    tag_list = list(df_keys["keys"])
    corpus = df_keys[purpose_description].tolist()#変更
    
    # Build Doc2Vec model
    model = build_doc2vec_model(corpus, tag_list, params)
    
    # Generate compound vectors
    compound_vec = add_vectors(tag_list, model)
    X_vec = np.array([compound_vec[i] for i in range(len(df_keys))])
    
    # Create index mapping
    index_mapping = create_index_mapping(len(df), invalid_indices)
    
    results = {}
    for category in categories:
        y = np.array([1 if i == category else 0 for i in df[category]])
        y_keys = np.array([1 if i == category else 0 for i in df_keys[category]])
        results[category] = evaluate_with_keys(category, X_vec, y, y_keys, lightgbm_model,index_mapping)

    return results

In [ ]:
# Tag Maccs keys
input_path = "data/train_df2.pkl"
# Generate MACCS fingerprints
maccs_features, invalid_indices = generate_maccs_fingerprints(df)
results_maccs = main_keys(input_path, maccs_features, invalid_indices, doc2vec_param, lightgbm_model, "description_gensim")

Invalid MACCS for index 183
Invalid MACCS for index 871
Invalid MACCS for index 872
Invalid MACCS for index 873
Number of compounds with valid MACCS keys: 3529


In [137]:
with open("result_tagchange/maccs_key.pkl", "wb") as f:
    pickle.dump(results_maccs, f)

In [121]:
categories = [
        'antioxidant', 'anti_inflammatory_agent', 'allergen', 'dye', 'toxin', 
        'flavouring_agent', 'agrochemical', 'volatile_oil', 'antibacterial_agent', 'insecticide'
    ]
li = []
for category, result in results_maccs.items():
    print(f"## {category} ##")
    print(results_maccs[category]['mcc']["mean_test"])
    li.append(results_maccs[category]['mcc']["mean_test"])
print("")
print(np.mean(li))

## antioxidant ##
0.5765570241633159
## anti_inflammatory_agent ##
0.5593128321034643
## allergen ##
0.6308256444903253
## dye ##
0.8905025070843096
## toxin ##
0.48499087811456076
## flavouring_agent ##
0.6515752867441196
## agrochemical ##
0.7168313079670604
## volatile_oil ##
0.770777646437359
## antibacterial_agent ##
0.4864654635652618
## insecticide ##
0.6650124924645456

0.6432851083134321


!!! pharmacore !!!

In [125]:
from rdkit.Chem.Pharm2D import Generate, Gobbi_Pharm2D
from tqdm import tqdm

In [126]:
def generate_pharmacophore(df: pd.DataFrame) -> Tuple[List[Optional[List[int]]], List[int]]:
    """
    Process pharmacophore features and identify invalid entries
    
    Args:
        df: DataFrame containing RDKit molecule objects in a column named 'ROMol'
        
    Returns:
        Tuple containing:
            - List of pharmacophore fingerprint bit indices (List[int]) or None for invalid entries
            - List of indices where pharmacophore generation failed
    """
    pharmacore_list = []
    for i in tqdm(df["ROMol"]):
        try:
            fp = Generate.Gen2DFingerprint(i, Gobbi_Pharm2D.factory)
            fp_bits = list(fp.GetOnBits())
            if len(fp_bits) == 0:
                pharmacore_list.append(None)
            else:
                pharmacore_list.append(fp_bits)
        except:
            print("Error")
            pharmacore_list.append(None)
            
    invalid_pharmacore_indices = []
    
    for idx, feature in enumerate(pharmacore_list):
        if feature is None:
            invalid_pharmacore_indices.append(idx)

    with open("pharmacore_list.pkl", "wb") as f:
        pickle.dump(pharmacore_list, f)

    return pharmacore_list, invalid_pharmacore_indices

In [133]:
input_path = "data/train_df2.pkl"
pharmacore_list, invalid_pharmacore_indices = generate_pharmacophore(df)

100%|██████████| 3533/3533 [03:36<00:00, 16.29it/s] 


In [ ]:
results_pharmacore = main_keys(input_path, pharmacore_list, invalid_pharmacore_indices, doc2vec_param, lightgbm_model, "description_gensim")

Number of compounds with valid keys: 3443


In [136]:
with open("result_tagchange/pharmacore.pkl", "wb") as f:
    pickle.dump(results_pharmacore, f)

!!! smiles Ngram !!!

In [131]:
from sklearn.feature_extraction.text import CountVectorizer

def smiles_to_ngrams(smiles_list: List[str], n: int) -> List[List[str]]:
    """
    Convert SMILES strings to n-grams
    
    Args:
        smiles_list: List of SMILES strings
        n: Integer specifying the n-gram size
        
    Returns:
        List of lists containing n-grams for each SMILES string
    """
    ngrams_list = []
    for smiles in smiles_list: 
        if len(smiles) < n:
            ngrams_list.append([smiles])
        else:
            ngrams_list.append([smiles[i:i+n] for i in range(len(smiles) - n + 1)])
    return ngrams_list

def make_ngramlist(df: pd.DataFrame, NAME: str, n: int = 3) -> List[List[int]]:
      
    # Generate n-grams from SMILES   
    smiles_list = list(df[NAME])
    ngrams_list = smiles_to_ngrams(smiles_list, n)

    # Convert n-grams to binary vectors
    vectorizer = CountVectorizer(binary=True, analyzer=lambda x: x)
    vec = vectorizer.fit_transform(ngrams_list)
  
    # Convert sparse vectors to index lists
    ngram_list = []
    for i in vec.toarray():
        li = []
        for j in range(len(i)):
            if i[j] == 1:
                li.append(j)
        ngram_list.append(li)
        
    return ngram_list

In [132]:
input_path = "data/train_df2.pkl"
with open(input_path, "rb") as f:
    df = pickle.load(f)
ngram_list = make_ngramlist(df, "smiles", n=3)
results_ngram = main(df, ngram_list, doc2vec_param, lightgbm_model, "description_gensim")

In [135]:
with open("result_tagchange/smiles_ngram.pkl", "wb") as f:
    pickle.dump(results_ngram, f)

In [134]:
categories = [
        'antioxidant', 'anti_inflammatory_agent', 'allergen', 'dye', 'toxin', 
        'flavouring_agent', 'agrochemical', 'volatile_oil', 'antibacterial_agent', 'insecticide'
    ]
li = []
for category, result in results_ngram.items():
    print(f"## {category} ##")
    print(results_ngram[category]['mcc']["mean_test"])
    li.append(results_ngram[category]['mcc']["mean_test"])
print("")
print(np.mean(li))

## antioxidant ##
0.5585740646341003
## anti_inflammatory_agent ##
0.5796338869808266
## allergen ##
0.6131707061195789
## dye ##
0.9146415442795559
## toxin ##
0.5249612720286082
## flavouring_agent ##
0.6449391313972289
## agrochemical ##
0.7253452068460022
## volatile_oil ##
0.7154934213434724
## antibacterial_agent ##
0.5129873809321751
## insecticide ##
0.6678227773643102

0.6457569391925859


In [16]:
input_path = "ECFP4096bit.pkl"
with open(input_path, "rb") as f:
    a = pickle.load(f)

In [18]:
categories = [
        'antioxidant', 'anti_inflammatory_agent', 'allergen', 'dye', 'toxin', 
        'flavouring_agent', 'agrochemical', 'volatile_oil', 'antibacterial_agent', 'insecticide'
    ]
li = []
for category, result in a.items():
    print(f"## {category} ##")
    print(a[category]['f1']["mean_test"])
    li.append(a[category]['f1']["mean_test"])
print("")
print(np.mean(li))

## antioxidant ##
0.7221319778859124
## anti_inflammatory_agent ##
0.7467746235518791
## allergen ##
0.6912898979130322
## dye ##
0.9385639376334233
## toxin ##
0.6287459557985874
## flavouring_agent ##
0.7306613370362036
## agrochemical ##
0.8155121280984631
## volatile_oil ##
0.8023856634836968
## antibacterial_agent ##
0.6956427669841374
## insecticide ##
0.7729619861371464

0.7544670274522483
